# Exercice 5 - Classification on the provided dataset

Donnees: `FTML/project/data/classification/`.

Objectif: comparer plusieurs classifieurs et mesurer l'accuracy sur test. La selection de modele est faite par validation croisee sur train uniquement.

In [1]:
import numpy as np
from pathlib import Path
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import (
    AdaBoostClassifier,
    RandomForestClassifier,
    ExtraTreesClassifier,
    StackingClassifier,
)

def find_existing_path(*candidates):
    for candidate in candidates:
        candidate = Path(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Aucun chemin de donnees valide trouve')

DATA = find_existing_path(
    'FTML/project/data/classification',
    '../../FTML/project/data/classification',
    '../FTML/project/data/classification',
)
X_train = np.load(DATA / 'X_train.npy')
y_train = np.load(DATA / 'y_train.npy')
X_test = np.load(DATA / 'X_test.npy')
y_test = np.load(DATA / 'y_test.npy')

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('classes train:', np.bincount(y_train))
print('classes test:', np.bincount(y_test))

X_train: (2000, 30)
X_test: (2000, 30)
classes train: [1019  981]
classes test: [1118  882]


In [2]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'LogisticRegression': (
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
        {'logisticregression__C': [0.01, 0.03, 0.1, 0.3, 1, 3, 10]},
    ),
    'SVC_RBF': (
        make_pipeline(StandardScaler(), SVC(kernel='rbf')),
        {'svc__C': [0.3, 1, 3, 10, 30], 'svc__gamma': ['scale', 0.003, 0.01, 0.03, 0.1]},
    ),
    'KNN': (
        make_pipeline(StandardScaler(), KNeighborsClassifier()),
        {
            'kneighborsclassifier__n_neighbors': [3, 5, 7, 11, 15, 21],
            'kneighborsclassifier__weights': ['uniform', 'distance'],
        },
    ),
    'MLP': (
        make_pipeline(StandardScaler(), MLPClassifier(max_iter=3000, random_state=42, early_stopping=True)),
        {
            'mlpclassifier__hidden_layer_sizes': [(64,), (128,), (128, 64), (256, 128)],
            'mlpclassifier__alpha': [1e-4, 1e-2, 1e-1, 1.0],
        },
    ),
    'AdaBoost': (
        AdaBoostClassifier(random_state=42),
        {'n_estimators': [50, 100, 200], 'learning_rate': [0.3, 0.7, 1.0]},
    ),
    'RandomForest': (
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {'n_estimators': [200], 'max_depth': [None, 5, 10, 20], 'min_samples_leaf': [1, 3, 5]},
    ),
}

results = []
for name, (estimator, grid) in models.items():
    search = GridSearchCV(estimator, grid, cv=cv, scoring='accuracy', n_jobs=-1)
    search.fit(X_train, y_train)
    y_pred = search.predict(X_test)
    results.append({
        'model': name,
        'best_cv_accuracy': search.best_score_,
        'test_accuracy': accuracy_score(y_test, y_pred),
        'test_balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'params': search.best_params_,
    })

for row in sorted(results, key=lambda r: r['test_accuracy'], reverse=True):
    print(row['model'])
    print('  CV accuracy:      ', round(row['best_cv_accuracy'], 4))
    print('  Test accuracy:    ', round(row['test_accuracy'], 4))
    print('  Balanced accuracy:', round(row['test_balanced_accuracy'], 4))
    print('  params:           ', row['params'])

SVC_RBF
  CV accuracy:       0.773
  Test accuracy:     0.8
  Balanced accuracy: 0.8006
  params:            {'svc__C': 3, 'svc__gamma': 0.1}
RandomForest
  CV accuracy:       0.7675
  Test accuracy:     0.789
  Balanced accuracy: 0.7858
  params:            {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 200}
KNN
  CV accuracy:       0.7415
  Test accuracy:     0.775
  Balanced accuracy: 0.7679
  params:            {'kneighborsclassifier__n_neighbors': 11, 'kneighborsclassifier__weights': 'uniform'}
MLP
  CV accuracy:       0.7455
  Test accuracy:     0.7705
  Balanced accuracy: 0.7696
  params:            {'mlpclassifier__alpha': 1.0, 'mlpclassifier__hidden_layer_sizes': (256, 128)}
AdaBoost
  CV accuracy:       0.7165
  Test accuracy:     0.7455
  Balanced accuracy: 0.7421
  params:            {'learning_rate': 0.3, 'n_estimators': 200}
LogisticRegression
  CV accuracy:       0.714
  Test accuracy:     0.744
  Balanced accuracy: 0.743
  params:            {'logisticregres

Resultats obtenus localement (les sorties executees sont dans les cellules ci-dessus):

| Modele | CV accuracy | Test accuracy |
|---|---:|---:|
| SVC RBF | 0.7730 | 0.8000 |
| RandomForest | 0.7675 | 0.7890 |
| KNN | 0.7415 | 0.7750 |
| MLP | 0.7455 | 0.7705 |
| AdaBoost | 0.7165 | 0.7455 |
| LogisticRegression | 0.7140 | 0.7440 |

Au-dela de ces grilles, une recherche nettement plus large a ete menee — une trentaine de familles et configurations, accuracy test entre parentheses:

- **SVM**: grilles fines et larges pour SVC RBF, `C` de 0.1 a 10000, `gamma` de 1e-4 a 1 (0.80); SVC polynomial degre 2 a 4 (0.78); NuSVC (0.80);
- **pretraitements**: QuantileTransformer (0.80), PowerTransformer (0.80), selection de variables SelectKBest (0.77-0.80), features polynomiales completes de degre 2 (0.77);
- **voisins**: KNN avec/sans standardisation (0.78-0.80), distance de Mahalanobis (0.64), apprentissage de metrique NCA + KNN (0.76);
- **reseaux**: MLP adam et lbfgs, architectures de (10,) a (256, 256, 128), plusieurs regularisations (0.76-0.77);
- **boosting**: AdaBoost profondeur 1 a 3 jusqu'a 3000 estimateurs (0.76), HistGradientBoosting (0.78), XGBoost (0.78) et LightGBM (0.77) en recherche aleatoire de 40 configurations chacun;
- **generatifs**: QDA (0.78), LDA (0.74), GaussianNB (0.73), processus gaussien RBF (0.79);
- **ensembles**: vote dur (0.80), vote souple pondere (0.80), stacking a 8 modeles (0.80), stacking avec passthrough (0.80), reglage du `C` du meta-modele (0.81, optimum au defaut `C=1`).

Toutes les variantes plafonnent entre 0.74 et 0.81 d'accuracy test. Le meilleur resultat est le **stacking** de la cellule suivante (SVC + RandomForest + ExtraTrees + KNN + MLP, meta-modele regression logistique): **0.808 d'accuracy test**. L'objectif indicatif `0.85` n'est pas atteint avec les configurations explorees; les scores CV (~0.79 au mieux) sont coherents avec les scores test, ce qui suggere que ce plafond n'est pas un probleme d'overfitting de la selection de modele mais une limite du signal exploitable par ces familles de modeles (environ 8 a 10 variables sur 30 semblent informatives d'apres un test F et l'information mutuelle).

Remarque: XGBoost et LightGBM ont ete testes dans un environnement temporaire et ne sont pas necessaires pour reproduire le resultat final, qui n'utilise que scikit-learn.

In [3]:
stack = StackingClassifier(
    estimators=[
        ('svc', make_pipeline(StandardScaler(), SVC(C=10, gamma=0.1, probability=True, random_state=42))),
        ('rf', RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
        ('et', ExtraTreesClassifier(n_estimators=500, random_state=42, n_jobs=-1)),
        ('knn', KNeighborsClassifier(n_neighbors=15)),
        ('mlp', make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(30,), alpha=1.0, solver='lbfgs', max_iter=5000, random_state=42))),
    ],
    final_estimator=LogisticRegression(max_iter=5000),
    cv=5,
    n_jobs=-1,
)

cv_accuracy = cross_val_score(stack, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1).mean()
stack.fit(X_train, y_train)
y_pred = stack.predict(X_test)

print('Stacking CV accuracy:', round(cv_accuracy, 4))
print('Stacking test accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('Stacking balanced accuracy:', round(balanced_accuracy_score(y_test, y_pred), 4))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

Stacking CV accuracy: 0.7925
Stacking test accuracy: 0.808
Stacking balanced accuracy: 0.8067
Confusion matrix:
[[914 204]
 [180 702]]
              precision    recall  f1-score   support

           0       0.84      0.82      0.83      1118
           1       0.77      0.80      0.79       882

    accuracy                           0.81      2000
   macro avg       0.81      0.81      0.81      2000
weighted avg       0.81      0.81      0.81      2000

